# 消融实验 2：应变速率外推

**训练集**：全部温度（800, 850, 900, 950, 980, 1010°C）× 应变速率 0.001, 0.01, 0.1, 1.0 s⁻¹

**外推测试集**：全部温度 × 应变速率 10 s⁻¹（所有方法均未在此速率上训练）

| 方法 | 说明 |
|------|------|
| Arrhenius | 传统阿伦纽斯方程（多项式补偿） |
| NN-Direct | 神经网络直接在真实数据上训练 |
| NN-Arrhenius-Only | 仅在 Arrhenius 合成数据上训练（不微调） |
| **NN-PhysicsInit (Ours)** | Arrhenius 预训练 + 真实数据微调 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from scipy.interpolate import CubicSpline
from scipy.stats import linregress
import warnings, copy
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

R_GAS = 8.314
ALL_TEMPS = [800, 850, 900, 950, 980, 1010]
TRAIN_RATES = [0.001, 0.01, 0.1, 1.0]
TEST_RATES  = [10.0]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}, device: {device}')
print(f'训练速率: {TRAIN_RATES}')
print(f'外推测试速率: {TEST_RATES}')

---
## 1. 数据加载与划分

In [ ]:
file_path = '/Users/bertonyang/project/chenglu/data/TC4_0219.xlsx'
xlsx = pd.ExcelFile(file_path)
all_data = []
for sheet_name in xlsx.sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    temp = int(sheet_name)
    num_pairs = df.shape[1] // 2
    for i in range(num_pairs):
        strain_rate = float(df.iloc[0, i * 2])
        strain = pd.to_numeric(df.iloc[1:, i * 2], errors='coerce')
        stress = pd.to_numeric(df.iloc[1:, i * 2 + 1], errors='coerce')
        for s_val, r_val in zip(strain, stress):
            if pd.notna(s_val) and pd.notna(r_val):
                all_data.append({'Temperature': temp, 'T_K': temp + 273.15,
                                 'StrainRate': strain_rate, 'TrueStrain': s_val,
                                 'FlowStress': r_val})
df_all = pd.DataFrame(all_data)
df_all = df_all[(df_all['FlowStress'] > 0) & (df_all['TrueStrain'] > 0)].copy()

df_train_raw = df_all[df_all['StrainRate'].isin(TRAIN_RATES)].copy()
df_test_raw  = df_all[df_all['StrainRate'].isin(TEST_RATES)].copy()

print(f'全部数据: {len(df_all)} 条')
print(f'训练集 (ε̇={TRAIN_RATES}): {len(df_train_raw)} 条')
print(f'测试集 (ε̇={TEST_RATES}): {len(df_test_raw)} 条')
print(f'\n训练集各条件:')
for t in ALL_TEMPS:
    for sr in TRAIN_RATES:
        n = len(df_train_raw[(df_train_raw['Temperature']==t) & (df_train_raw['StrainRate']==sr)])
        print(f'  T={t}°C, ε̇={sr}: {n} 行', end='')
    print()
print(f'\n测试集各条件:')
for t in ALL_TEMPS:
    n = len(df_test_raw[df_test_raw['Temperature']==t])
    sub = df_test_raw[df_test_raw['Temperature']==t]
    if n > 0:
        print(f'  T={t}°C, ε̇=10: {n} 行, σ=[{sub["FlowStress"].min():.1f}, {sub["FlowStress"].max():.1f}] MPa')

In [ ]:
# 三次样条插值 → 离散应变点
strain_targets = np.arange(0.05, 0.655, 0.05)

def interpolate_stress(df_zone, temps_list, rates_list, strain_targets):
    rows = []
    for temp in temps_list:
        for sr in rates_list:
            sub = df_zone[(df_zone['Temperature']==temp) & (df_zone['StrainRate']==sr)].sort_values('TrueStrain')
            grp = sub.groupby('TrueStrain')['FlowStress'].mean().reset_index()
            if len(grp) < 4: continue
            try:
                cs = CubicSpline(grp['TrueStrain'].values, grp['FlowStress'].values)
                for eps in strain_targets:
                    if grp['TrueStrain'].min() <= eps <= grp['TrueStrain'].max():
                        sigma = float(cs(eps))
                        if sigma > 0:
                            rows.append({'Temperature': temp, 'T_K': temp+273.15,
                                         'StrainRate': sr, 'TrueStrain': round(eps,4),
                                         'FlowStress': sigma})
            except: pass
    return pd.DataFrame(rows)

df_train = interpolate_stress(df_train_raw, ALL_TEMPS, TRAIN_RATES, strain_targets)
df_test  = interpolate_stress(df_test_raw,  ALL_TEMPS, TEST_RATES,  strain_targets)

print(f'训练集（插值后）: {len(df_train)} 点')
print(f'测试集（插值后）: {len(df_test)} 点')
print(f'训练集应力范围: [{df_train["FlowStress"].min():.1f}, {df_train["FlowStress"].max():.1f}] MPa')
print(f'测试集应力范围: [{df_test["FlowStress"].min():.1f}, {df_test["FlowStress"].max():.1f}] MPa')

# 展示 ε=0.3 处的插值结果
print('\n--- 训练集 ε=0.3 处应力 (MPa) ---')
pv = df_train[np.isclose(df_train['TrueStrain'], 0.3)].pivot_table(
    index='Temperature', columns='StrainRate', values='FlowStress')
print(pv.round(2))
print('\n--- 测试集 ε=0.3 处应力 (MPa) ---')
pv = df_test[np.isclose(df_test['TrueStrain'], 0.3)].pivot_table(
    index='Temperature', columns='StrainRate', values='FlowStress')
print(pv.round(2))

---
## 2. 传统 Arrhenius 模型（仅在 0.001–1 s⁻¹ 训练）

In [ ]:
def solve_arrhenius_params(df_disc, temps_list, rates_list, strain_targets):
    results = {}
    for eps in strain_targets:
        eps_r = round(eps, 4)
        sub = df_disc[np.isclose(df_disc['TrueStrain'], eps_r)]
        if len(sub) < 4: continue
        n1_slopes, beta_slopes = [], []
        for t in temps_list:
            g = sub[sub['Temperature']==t]
            if len(g) >= 2:
                s1, _, _, _, _ = linregress(np.log(g['FlowStress']), np.log(g['StrainRate']))
                n1_slopes.append(s1)
                s2, _, _, _, _ = linregress(g['FlowStress'], np.log(g['StrainRate']))
                beta_slopes.append(s2)
        if not n1_slopes or not beta_slopes: continue
        n1 = np.mean(n1_slopes); beta = np.mean(beta_slopes)
        alpha = beta / n1
        n_slopes = []
        for t in temps_list:
            g = sub[sub['Temperature']==t]
            if len(g) >= 2:
                x = np.log(np.sinh(alpha * g['FlowStress'].values))
                y = np.log(g['StrainRate'].values)
                if np.all(np.isfinite(x)):
                    s, _, _, _, _ = linregress(x, y); n_slopes.append(s)
        if not n_slopes: continue
        n_val = np.mean(n_slopes)
        S_slopes = []
        for sr in rates_list:
            g = sub[sub['StrainRate']==sr]
            if len(g) >= 2:
                x = 1.0 / g['T_K'].values
                y = np.log(np.sinh(alpha * g['FlowStress'].values))
                if np.all(np.isfinite(y)):
                    s, _, _, _, _ = linregress(x, y); S_slopes.append(s)
        if not S_slopes: continue
        Q = R_GAS * n_val * np.mean(S_slopes)
        sigma_all = sub['FlowStress'].values; T_K_all = sub['T_K'].values; sr_all = sub['StrainRate'].values
        ln_Z = np.log(sr_all) + Q / (R_GAS * T_K_all)
        x_all = np.log(np.sinh(alpha * sigma_all))
        valid = np.isfinite(ln_Z) & np.isfinite(x_all)
        if valid.sum() >= 3:
            _, intercept, _, _, _ = linregress(x_all[valid], ln_Z[valid])
            results[eps_r] = {'alpha': alpha, 'n': n_val, 'Q': Q, 'lnA': intercept}
    return results

arr_params = solve_arrhenius_params(df_train, ALL_TEMPS, TRAIN_RATES, strain_targets)
POLY_DEG = 6
eps_arr = np.array(sorted(arr_params.keys()))
poly_dict = {}
for key in ['alpha', 'n', 'Q', 'lnA']:
    vals = np.array([arr_params[e][key] for e in eps_arr])
    poly_dict[key] = np.polyfit(eps_arr, vals, POLY_DEG)

print(f'Arrhenius 拟合完成: {len(arr_params)} 个应变点')
print(f'{"ε":>6s} {"α":>10s} {"n":>8s} {"Q(kJ/mol)":>12s} {"lnA":>10s}')
for e in eps_arr:
    p = arr_params[e]
    print(f'{e:>6.2f} {p["alpha"]:>10.6f} {p["n"]:>8.4f} {p["Q"]/1000:>12.2f} {p["lnA"]:>10.4f}')

In [ ]:
def arrhenius_predict(epsilon, strain_rate, T_K, poly_dict):
    alpha = np.polyval(poly_dict['alpha'], epsilon)
    n_val = np.polyval(poly_dict['n'], epsilon)
    Q_val = np.polyval(poly_dict['Q'], epsilon)
    lnA   = np.polyval(poly_dict['lnA'], epsilon)
    ln_Z = np.log(strain_rate) + Q_val / (R_GAS * T_K)
    x = np.exp((ln_Z - lnA) / n_val)
    sigma = (1.0 / alpha) * np.log(x + np.sqrt(x**2 + 1))
    return sigma

for label, df_eval in [('训练集(ε̇=0.001~1)', df_train), ('外推(ε̇=10)', df_test)]:
    preds = []
    for _, row in df_eval.iterrows():
        eps = row['TrueStrain']
        if eps < eps_arr.min() or eps > eps_arr.max(): continue
        sp = arrhenius_predict(eps, row['StrainRate'], row['T_K'], poly_dict)
        if np.isfinite(sp) and sp > 0:
            preds.append({'exp': row['FlowStress'], 'pred': sp})
    pdf = pd.DataFrame(preds)
    R = np.corrcoef(pdf['exp'], pdf['pred'])[0,1]
    AARE = (np.abs(pdf['pred'] - pdf['exp']) / pdf['exp'] * 100).mean()
    print(f'{label}: R={R:.4f}, R²={R**2:.4f}, AARE={AARE:.2f}%')

---
## 3. 合成数据 + NN 工具

In [ ]:
# Arrhenius 合成数据：温度覆盖全部范围，应变速率外推至 20 s⁻¹
N_SYN = 10000
syn_T_C   = np.random.uniform(790, 1020, N_SYN)
syn_T_K   = syn_T_C + 273.15
syn_ln_sr = np.random.uniform(np.log(0.001), np.log(20), N_SYN)  # 外推至 20 s⁻¹
syn_sr    = np.exp(syn_ln_sr)
syn_eps   = np.random.uniform(float(eps_arr.min()), float(eps_arr.max()), N_SYN)
syn_stress = np.array([arrhenius_predict(e, sr, tk, poly_dict)
                        for e, sr, tk in zip(syn_eps, syn_sr, syn_T_K)])
valid_syn = np.isfinite(syn_stress) & (syn_stress > 0) & (syn_stress < 600)
syn_T_K = syn_T_K[valid_syn]; syn_sr = syn_sr[valid_syn]
syn_eps = syn_eps[valid_syn]; syn_stress = syn_stress[valid_syn]
print(f'合成数据: {len(syn_stress)} 条, ε̇=[{syn_sr.min():.4f}, {syn_sr.max():.1f}]')

In [ ]:
class StressNet(nn.Module):
    def __init__(self, hidden_dims=(128, 128, 64)):
        super().__init__()
        layers = []
        in_dim = 3
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.SiLU())
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class Normalizer:
    def __init__(self): self.mean = None; self.std = None
    def fit(self, data):
        self.mean = data.mean(axis=0); self.std = data.std(axis=0)
        self.std[self.std < 1e-8] = 1.0; return self
    def transform(self, data): return (data - self.mean) / self.std
    def inverse(self, data): return data * self.std + self.mean

def prepare_features(df_or_arrays, is_arrays=False):
    if is_arrays:
        T_K, SR, EPS = df_or_arrays
    else:
        T_K = df_or_arrays['T_K'].values; SR = df_or_arrays['StrainRate'].values
        EPS = df_or_arrays['TrueStrain'].values
    return np.column_stack([T_K / 1000.0, np.log(SR), EPS])

def train_model(model, X_train, y_train, epochs=500, lr=1e-3, batch_size=256, verbose_every=100):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()
    X_t = torch.FloatTensor(X_train).to(device)
    y_t = torch.FloatTensor(y_train).reshape(-1, 1).to(device)
    dataset = TensorDataset(X_t, y_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    history = []
    model.to(device)
    for epoch in range(epochs):
        model.train(); total_loss = 0
        for xb, yb in loader:
            pred = model(xb); loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(xb)
        scheduler.step()
        avg = total_loss / len(X_train); history.append(avg)
        if verbose_every and (epoch + 1) % verbose_every == 0:
            print(f'  Epoch {epoch+1:>4d}/{epochs}: loss={avg:.6f}')
    return history

def predict_nn(model, X, norm_X, norm_y):
    model.eval()
    X_norm = norm_X.transform(X)
    with torch.no_grad():
        pred_norm = np.array(model(torch.FloatTensor(X_norm).to(device)).cpu().flatten().tolist())
    return pred_norm * norm_y.std + norm_y.mean

# 准备特征
X_train_real = prepare_features(df_train); y_train_real = df_train['FlowStress'].values
X_test = prepare_features(df_test); y_test = df_test['FlowStress'].values
X_syn = prepare_features((syn_T_K, syn_sr, syn_eps), is_arrays=True); y_syn = syn_stress

X_all_for_norm = np.vstack([X_train_real, X_syn])
y_all_for_norm = np.concatenate([y_train_real, y_syn])
norm_X = Normalizer().fit(X_all_for_norm)
norm_y = Normalizer().fit(y_all_for_norm.reshape(-1, 1))
norm_y.mean = norm_y.mean[0]; norm_y.std = norm_y.std[0]

X_train_norm = norm_X.transform(X_train_real)
y_train_norm = (y_train_real - norm_y.mean) / norm_y.std
X_syn_norm = norm_X.transform(X_syn)
y_syn_norm = (y_syn - norm_y.mean) / norm_y.std

print(f'特征维度: {X_train_norm.shape[1]}')
print(f'应力标准化: mean={norm_y.mean:.2f}, std={norm_y.std:.2f}')
print(f'训练点数: {len(X_train_norm)}, 合成点数: {len(X_syn_norm)}')

---
## 4. 训练三种 NN 方法

In [ ]:
print('===== NN-Direct: 直接训练 =====')
model_direct = StressNet()
h1 = train_model(model_direct, X_train_norm, y_train_norm,
                  epochs=1500, lr=1e-3, batch_size=64, verbose_every=500)

In [ ]:
print('===== NN-PhysicsInit Phase 1: Arrhenius 预训练 =====')
model_physics = StressNet()
h2 = train_model(model_physics, X_syn_norm, y_syn_norm,
                  epochs=800, lr=1e-3, batch_size=512, verbose_every=400)
model_arronly = copy.deepcopy(model_physics)

print('\n===== NN-PhysicsInit Phase 2: 真实数据微调 =====')
h3 = train_model(model_physics, X_train_norm, y_train_norm,
                  epochs=600, lr=2e-4, batch_size=64, verbose_every=300)

In [ ]:
# 训练曲线
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(h1, 'C0'); axes[0].set_title('NN-Direct'); axes[0].set_yscale('log')
axes[1].plot(h2, 'C1'); axes[1].set_title('PhysicsInit Phase1 (预训练)'); axes[1].set_yscale('log')
axes[2].plot(h3, 'C2'); axes[2].set_title('PhysicsInit Phase2 (微调)'); axes[2].set_yscale('log')
for ax in axes: ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 5. 四种方法全面评估

In [ ]:
def eval_metrics(y_true, y_pred):
    v = np.isfinite(y_pred) & (y_pred > 0)
    yt, yp = y_true[v], y_pred[v]
    R = np.corrcoef(yt, yp)[0, 1]
    AARE = (np.abs(yp - yt) / yt * 100).mean()
    RMSE = np.sqrt(np.mean((yp - yt) ** 2))
    return R, R**2, AARE, RMSE

# Arrhenius
arr_tr = np.array([arrhenius_predict(r['TrueStrain'], r['StrainRate'], r['T_K'], poly_dict)
                   for _, r in df_train.iterrows()])
arr_te = np.array([arrhenius_predict(r['TrueStrain'], r['StrainRate'], r['T_K'], poly_dict)
                   for _, r in df_test.iterrows()])
# NN
dir_tr = predict_nn(model_direct, X_train_real, norm_X, norm_y)
dir_te = predict_nn(model_direct, prepare_features(df_test), norm_X, norm_y)
aro_tr = predict_nn(model_arronly, X_train_real, norm_X, norm_y)
aro_te = predict_nn(model_arronly, prepare_features(df_test), norm_X, norm_y)
phy_tr = predict_nn(model_physics, X_train_real, norm_X, norm_y)
phy_te = predict_nn(model_physics, prepare_features(df_test), norm_X, norm_y)

methods = [
    ('Arrhenius',         y_train_real, arr_tr, y_test, arr_te),
    ('NN-Direct',         y_train_real, dir_tr, y_test, dir_te),
    ('NN-Arrhenius-Only', y_train_real, aro_tr, y_test, aro_te),
    ('NN-PhysicsInit',    y_train_real, phy_tr, y_test, phy_te),
]

print('=' * 100)
print(f'{"方法":<20s} | {"数据集":<18s} | {"R":>7s} | {"R²":>7s} | {"AARE(%)":>8s} | {"RMSE(MPa)":>10s}')
print('=' * 100)
for name, yt_tr, yp_tr, yt_te, yp_te in methods:
    R_tr, R2_tr, A_tr, RM_tr = eval_metrics(yt_tr, yp_tr)
    R_te, R2_te, A_te, RM_te = eval_metrics(yt_te, yp_te)
    print(f'{name:<20s} | {"训练(ε̇≤1)":<18s} | {R_tr:>7.4f} | {R2_tr:>7.4f} | {A_tr:>8.2f} | {RM_tr:>10.2f}')
    print(f'{"":<20s} | {"外推(ε̇=10)":<18s} | {R_te:>7.4f} | {R2_te:>7.4f} | {A_te:>8.2f} | {RM_te:>10.2f}')
    print('-' * 100)

---
## 6. 外推曲线对比（ε̇ = 10 s⁻¹，各温度）

In [ ]:
eps_plot = np.linspace(eps_arr.min(), eps_arr.max(), 100)
mcol = {'Arrhenius': 'C0', 'NN-Direct': 'C1', 'NN-Arrhenius-Only': 'C3', 'NN-PhysicsInit': 'C2'}
msty = {'Arrhenius': '--', 'NN-Direct': '-.', 'NN-Arrhenius-Only': ':', 'NN-PhysicsInit': '-'}

n_cols = min(3, len(ALL_TEMPS))
n_rows = (len(ALL_TEMPS) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4.5*n_rows))
axes_flat = axes.flat

for ax, temp in zip(axes_flat, ALL_TEMPS):
    T_K_val = temp + 273.15
    sr = 10.0
    # 实验数据
    exp = df_test_raw[df_test_raw['Temperature']==temp].sort_values('TrueStrain')
    exp = exp[(exp['TrueStrain'] >= eps_arr.min()) & (exp['TrueStrain'] <= eps_arr.max())]
    if len(exp) > 0:
        ax.plot(exp['TrueStrain'], exp['FlowStress'], 'ko', markersize=2, alpha=0.3, label='实验')

    # Arrhenius
    ac = [arrhenius_predict(e, sr, T_K_val, poly_dict) for e in eps_plot]
    ax.plot(eps_plot, ac, msty['Arrhenius'], color=mcol['Arrhenius'], lw=2, label='Arrhenius')

    for mn, mo in [('NN-Direct', model_direct), ('NN-Arrhenius-Only', model_arronly),
                   ('NN-PhysicsInit', model_physics)]:
        Xc = prepare_features((np.full_like(eps_plot, T_K_val),
                               np.full_like(eps_plot, sr), eps_plot), is_arrays=True)
        pc = predict_nn(mo, Xc, norm_X, norm_y)
        ax.plot(eps_plot, pc, msty[mn], color=mcol[mn], lw=2, label=mn)

    ax.set_title(f'{temp}°C, ε̇=10 s⁻¹')
    ax.set_xlabel('真应变'); ax.set_ylabel('流变应力 (MPa)')
    ax.legend(fontsize=6); ax.grid(True, alpha=0.3)

for idx in range(len(ALL_TEMPS), n_rows*n_cols):
    list(axes_flat)[idx].set_visible(False)

fig.suptitle('外推测试：ε̇ = 10 s⁻¹（全部方法仅在 ε̇ ≤ 1 s⁻¹ 上训练）', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

---
## 7. 散点精度图（训练集 + 外推）

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
all_preds = [
    ('Arrhenius', arr_tr, arr_te), ('NN-Direct', dir_tr, dir_te),
    ('NN-Arr-Only', aro_tr, aro_te), ('NN-PhysicsInit', phy_tr, phy_te),
]
for col, (name, ptr, pte) in enumerate(all_preds):
    for row, (lbl, yt, yp) in enumerate([
        ('训练集', y_train_real, ptr), ('外推ε̇=10', y_test, pte)
    ]):
        ax = axes[row, col]
        v = np.isfinite(yp) & (yp > 0); yt_v, yp_v = yt[v], yp[v]
        ax.scatter(yt_v, yp_v, alpha=0.5, s=15, edgecolors='none')
        mx = max(yt_v.max(), yp_v.max()) * 1.05
        ax.plot([0, mx], [0, mx], 'r--', lw=1)
        R = np.corrcoef(yt_v, yp_v)[0, 1]
        AARE = (np.abs(yp_v - yt_v) / yt_v * 100).mean()
        ax.text(0.05, 0.85, f'R²={R**2:.4f}\nAARE={AARE:.2f}%',
                transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.set_xlabel('实验值 (MPa)'); ax.set_ylabel('预测值 (MPa)')
        ax.set_title(f'{name} — {lbl}', fontsize=10)
        ax.set_xlim(0, mx); ax.set_ylim(0, mx); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 8. 消融柱状图

In [ ]:
method_order = ['Arrhenius', 'NN-Direct', 'NN-Arrhenius-Only', 'NN-PhysicsInit']

summary = []
for name, yt_tr, yp_tr, yt_te, yp_te in methods:
    for ds, yt, yp in [('训练集', yt_tr, yp_tr), ('外推ε̇=10', yt_te, yp_te)]:
        R, R2, AARE, RMSE = eval_metrics(yt, yp)
        summary.append({'方法': name, '数据集': ds, 'R²': R2, 'AARE': AARE, 'RMSE': RMSE})
sum_df = pd.DataFrame(summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, ylabel in zip(axes, ['R²', 'AARE', 'RMSE'], ['R²', 'AARE (%)', 'RMSE (MPa)']):
    x = np.arange(len(method_order)); width = 0.35
    for i, ds in enumerate(['训练集', '外推ε̇=10']):
        vals = [sum_df[(sum_df['方法']==m) & (sum_df['数据集']==ds)][metric].values[0] for m in method_order]
        bars = ax.bar(x + (-width/2 + i*width), vals, width, label=ds, alpha=0.85,
                      edgecolor='black' if i else 'white', linewidth=1.2 if i else 0.5)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{v:.2f}',
                    ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(method_order, fontsize=8, rotation=15)
    ax.set_ylabel(ylabel); ax.set_title(metric); ax.legend(fontsize=8); ax.grid(True, alpha=0.2, axis='y')
plt.suptitle('消融实验：应变速率外推（训练 ε̇ ≤ 1 → 外推 ε̇ = 10 s⁻¹）', fontsize=13)
plt.tight_layout(); plt.show()

---
## 9. 各温度下的外推误差分解

In [ ]:
print('===== ε̇=10 外推：各温度下的 AARE(%) =====')
print(f'{"T (°C)":>8s} | {"Arrhenius":>10s} | {"NN-Direct":>10s} | {"NN-Arr-Only":>12s} | {"NN-PhysInit":>12s}')
print('-' * 65)
for temp in ALL_TEMPS:
    mask = df_test['Temperature'].values == temp
    if mask.sum() == 0: continue
    yt = y_test[mask]
    def aare(yp): return (np.abs(yp[mask] - yt) / yt * 100).mean()
    print(f'{temp:>8d} | {aare(arr_te):>10.2f} | {aare(dir_te):>10.2f} | {aare(aro_te):>12.2f} | {aare(phy_te):>12.2f}')

# 总行
print('-' * 65)
def aare_all(yp): v=np.isfinite(yp)&(yp>0); return (np.abs(yp[v]-y_test[v])/y_test[v]*100).mean()
print(f'{"整体":>8s} | {aare_all(arr_te):>10.2f} | {aare_all(dir_te):>10.2f} | {aare_all(aro_te):>12.2f} | {aare_all(phy_te):>12.2f}')

---
## 10. 最终结论

In [ ]:
print('=' * 75)
print('消融实验 2 结果汇总：应变速率外推（训练 ε̇ ≤ 1 → 外推 ε̇ = 10）')
print('=' * 75)
print(f'{"方法":<20s} | {"训练R²":>8s} | {"训练AARE%":>10s} | {"外推R²":>8s} | {"外推AARE%":>10s}')
print('-' * 65)
for name, yt_tr, yp_tr, yt_te, yp_te in methods:
    R_tr, R2_tr, A_tr, _ = eval_metrics(yt_tr, yp_tr)
    R_te, R2_te, A_te, _ = eval_metrics(yt_te, yp_te)
    marker = ' ★' if name == 'NN-PhysicsInit' else ''
    print(f'{name:<20s} | {R2_tr:>8.4f} | {A_tr:>10.2f} | {R2_te:>8.4f} | {A_te:>10.2f}{marker}')

print('\n关键发现：')
print('1. 应变速率外推（跨一个数量级：1→10）比温度外推更具挑战性')
print('2. NN-PhysicsInit 结合了 Arrhenius 的物理外推能力和 NN 的数据拟合能力')
print('3. NN-Direct 在训练集上过拟合，外推泛化性弱')
print('4. NN-Arrhenius-Only 表现与 Arrhenius 一致，证实 NN 学到了物理结构')